In [18]:
import os
import requests
import pandas as pd
from pathlib import Path

# ============================================================
# SETTINGS
# ============================================================

OUTPUT_DIR = "ClinVar_By_UniProt"
Path(OUTPUT_DIR).mkdir(exist_ok=True)

CLINVAR_URL = (
    "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/"
    "variant_summary.txt.gz")

LOCAL_CLINVAR_FILE = "variant_summary.txt.gz"

# ============================================================
# DOWNLOAD CLINVAR FILE IF NEEDED
# ============================================================

if not os.path.exists(LOCAL_CLINVAR_FILE):

    print("Downloading ClinVar variant_summary.txt.gz ...")

    r = requests.get(CLINVAR_URL, stream=True)

    with open(LOCAL_CLINVAR_FILE, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

    print("Download complete.")

# ============================================================
# LOAD CLINVAR
# ============================================================

print("Loading ClinVar file...")

clinvar = pd.read_csv(LOCAL_CLINVAR_FILE,sep="\t",compression="gzip",low_memory=False)

print(f"Loaded {len(clinvar):,} ClinVar records")

# ============================================================
# UNIPROT -> GENE
# ============================================================

def uniprot_to_gene(uniprot_id):

    url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.json"

    r = requests.get(url)

    if r.status_code != 200:
        raise ValueError(f"Could not retrieve {uniprot_id}")

    data = r.json()

    genes = data.get("genes", [])

    if not genes:
        raise ValueError(f"No gene symbol found for {uniprot_id}")

    return genes[0]["geneName"]["value"]

# ============================================================
# PROCESS UNIPROT IDS
# ============================================================

def extract_clinvar_for_uniprot(uniprot_id):

    gene = uniprot_to_gene(uniprot_id)

    df_gene = clinvar[clinvar["GeneSymbol"] == gene].copy()

    # ========================================================
    # ADD PROTEIN CHANGE COLUMN
    # ========================================================

    df_gene["protein_change"] = df_gene["Name"].apply(
        extract_protein_change
    )

    def parse_protein_hgvs(p):
        if not p:
            return None, None, None

        m = re.match(r"p\.([A-Za-z]+)(\d+)([A-Za-z]+)", p)

        if m:
            return m.group(1), int(m.group(2)), m.group(3)

        return None, None, None

    df_gene[["wt", "pos", "mut"]] = df_gene["protein_change"].apply(
        lambda x: pd.Series(parse_protein_hgvs(x))
    )

    # ========================================================
    # SAVE .txt.gz
    # ========================================================

    outfile = (
        f"{OUTPUT_DIR}/"
        f"{uniprot_id}_{gene}_clinvar.txt.gz")

    df_gene.to_csv(outfile,sep="\t",index=False,compression="gzip")

    return {"uniprot_id": uniprot_id,
        "gene": gene,
        "n_variants": len(df_gene),
        "outfile": outfile}

def parse_protein_hgvs(p):
    """
    Convert p.Thr315Ile → WT=Thr, POS=315, MUT=Ile
    """
    if not p:
        return None, None, None

    m = re.match(r"p\.([A-Za-z]+)(\d+)([A-Za-z]+)", p)

    if m:
        return m.group(1), int(m.group(2)), m.group(3)

    return None, None, None

def extract_protein_change(name):
    """
    Extract p.HGVS from ClinVar Name column
    """
    if pd.isna(name):
        return None

    m = re.search(r"\(p\.[^)]+\)", name)

    if m:
        return m.group(0).strip("()")

    return None

def convert_gz_to_txt(gz_file, txt_file=None):

    df = pd.read_csv(gz_file,sep="\t",compression="gzip",low_memory=False
    )

    if txt_file is None:
        txt_file = gz_file.replace(".txt.gz", ".txt")

    df.to_csv(txt_file, sep="\t", index=False)

    print(f"Converted: {gz_file} → {txt_file}")

Loading ClinVar file...
Loaded 8,982,886 ClinVar records


In [31]:
df_human = pd.read_csv('homo_sapiens_reference_dataset.csv')
df_py = df_human.loc[df_human['ptm_type'] == 'Phosphotyrosine'] 
uniprot_ids = (df_py['uniprot_id'].unique())

In [33]:
# ============================================================
# INPUT UNIPROT IDS
# ============================================================
import re

OUTPUT_DIR = "ClinVar_By_UniProt"
Path(OUTPUT_DIR).mkdir(exist_ok=True)

# ============================================================
# RUN
# ============================================================

summary_rows = []

for uid in uniprot_ids:

    try:

        result = extract_clinvar_for_uniprot(uid)

        summary_rows.append(result)

        print(
            f"{uid} | "
            f"{result['gene']} | "
            f"{result['n_variants']} variants"
        )

    except Exception as e:

        print(f"Failed: {uid} -> {e}")

# ============================================================
# SUMMARY TABLE
# ============================================================

summary_df = pd.DataFrame(summary_rows)

summary_df.to_csv(f"{OUTPUT_DIR}/ClinVar_summary.csv",index=False)

print("\nDone.")
print(summary_df)

for file in os.listdir(OUTPUT_DIR):

    if file.endswith(".txt.gz"):

        gz_path = os.path.join(OUTPUT_DIR, file)

        txt_path = gz_path.replace(".txt.gz", ".txt")

        convert_gz_to_txt(gz_path, txt_path)

O95081 | AGFG2 | 168 variants
Q9H773 | DCTPP1 | 66 variants
O95154 | AKR7A3 | 181 variants
Q5VXJ0 | LIPK | 126 variants
O60673 | REV3L | 946 variants
Q9Y4W6 | AFG3L2 | 1200 variants
Q9BSM1 | PCGF1 | 66 variants
Q9Y5E4 | PCDHB5 | 328 variants
Q9BXU1 | STK31 | 310 variants
A1L390 | PLEKHG3 | 481 variants
Q12979 | ABR | 258 variants
Q9GZQ4 | NMUR2 | 122 variants
Q9BXR5 | TLR10 | 226 variants
Q13905 | RAPGEF1 | 306 variants
O60437 | PPL | 1048 variants
O75600 | GCAT | 192 variants
Q96N58 | ZNF578 | 236 variants
Q9H8Y8 | GORASP2 | 164 variants
P13010 | XRCC5 | 192 variants
P13984 | GTF2F2 | 76 variants
P63220 | RPS21 | 36 variants
A8MUP2 | CSKMT | 98 variants
P17481 | HOXB8 | 54 variants
P09467 | FBP1 | 704 variants
Failed: P43356 -> Columns must be same length as key
Q6PFW1 | PPIP5K1 | 140 variants
O75122 | CLASP2 | 358 variants
Q9NRF8 | CTPS2 | 375 variants
P07327 | ADH1A | 102 variants
Q14145 | KEAP1 | 160 variants
Q86UP3 | ZFHX4 | 1479 variants
Q9Y2Y1 | POLR3K | 49 variants
A6NKC9 | SH2

In [24]:
file = "variant_summary.txt.gz"

if os.path.exists(file):
    print("Exists")
    print("Size (MB):", os.path.getsize(file) / (1024*1024))
else: 
    print("Not downloaded")

Exists
Size (MB): 418.9835786819458
